### Phase 3 — Gold Layer: `gold.fct_flights`
Atomic-grain (flight-level) curated fact table, built from `silver.flights`.
This is the Genie-facing table: business-friendly columns, DQ-internal fields dropped or simplified, joins to `dim_date` / `dim_carrier` / `dim_airport` (built in a later step) via natural keys.
A separate aggregate table (`gold.agg_daily_carrier_route_performance`) is built from this table afterward, for dashboard performance.

In [0]:
spark.sql("USE CATALOG airline_analytics")
spark.sql("CREATE SCHEMA IF NOT EXISTS gold")

### Building Fact Table

##### Step 1: Pilot on a single year

In [0]:
PILOT_YEAR = 2015

df = spark.table("silver.flights").filter(f"year(flight_date) = {PILOT_YEAR}")
print(f"Pilot row count: {df.count():,}")
df.printSchema()

##### Step 2a: Transform pilot year into fct_flights shape

In [0]:
from pyspark.sql import functions as F

fct_flights_pilot = (df.withColumn("has_dq_warning", F.size("_dq_warnings") > 0)
    .withColumn("air_time_unreliable",F.array_contains("_dq_warnings", "AIRTIME_ELAPSED_MISMATCH"))
    .withColumnRenamed("_source_system", "data_source")
    .withColumn("_gold_processed_at", F.current_timestamp())
    .drop("_ingestion_timestamp", "_silver_processed_at", "_dq_warnings")
)

print(f"Pilot fct_flights row count: {fct_flights_pilot.count():,}")

##### Step 2b: Null percentage audit before finalizing columns

In [0]:
total = fct_flights_pilot.count()

null_pct = fct_flights_pilot.select([(F.round(F.sum(F.col(c).isNull().cast("int")) / total * 100, 1)).alias(c)
    for c in fct_flights_pilot.columns])

# Transpose for readability
null_pct_pd = null_pct.toPandas().T.reset_index()
null_pct_pd.columns = ["column", "null_pct"]
null_pct_pd = null_pct_pd.sort_values("null_pct", ascending=False)

display(null_pct_pd)

##### Step 2c: Finalize fct_flights column set

In [0]:
fct_flights_pilot = (fct_flights_pilot
                     .drop("cancellation_code", "crs_dep_hour", "dep_delay_min_pos", "arr_delay_min_pos"))

print(f"Final column count: {len(fct_flights_pilot.columns)}")

##### Step 2d: Feature engineering — flight-level derived columns

In [0]:
fct_flights_pilot = (
    fct_flights_pilot.withColumn("route", F.concat_ws("-", F.col("origin"), F.col("dest")))
    .withColumn("departure_time_block",
        F.when(F.floor(F.col("crs_dep_time_hhmm") / 100).between(0, 5), "Early Morning")
         .when(F.floor(F.col("crs_dep_time_hhmm") / 100).between(6, 11), "Morning")
         .when(F.floor(F.col("crs_dep_time_hhmm") / 100).between(12, 16), "Afternoon")
         .when(F.floor(F.col("crs_dep_time_hhmm") / 100).between(17, 20), "Evening")
         .when(F.floor(F.col("crs_dep_time_hhmm") / 100).between(21, 23), "Night")
         .otherwise(None)
    )
    .withColumn("haul_length", F.when(F.col("distance_mi") < 500, "Short-haul")
         .when(F.col("distance_mi") < 1500, "Medium-haul")
         .otherwise("Long-haul")
    )
)

print(f"Final column count: {len(fct_flights_pilot.columns)}")
fct_flights_pilot.select("route", "departure_time_block", "haul_length").show(5)

##### Step 3: Write pilot year to gold.fct_flights

In [0]:
(fct_flights_pilot.write.format("delta").mode("overwrite")
 .clusterBy("flight_date", "carrier_code")
 .saveAsTable("gold.fct_flights")
)

result = spark.sql("SELECT COUNT(*) AS row_count FROM gold.fct_flights")
result.show()

In [0]:
spark.sql("DESCRIBE TABLE gold.fct_flights").show(60, truncate=False)

##### Step 4: Backfill gold.fct_flights in 3-year batches

In [0]:
def build_fct_flights(source_df):
    """Applies the full Silver -> Gold transformation validated in the pilot."""
    return (source_df.withColumn("has_dq_warning", F.size("_dq_warnings") > 0)
            .withColumn("air_time_unreliable", F.array_contains("_dq_warnings", "AIRTIME_ELAPSED_MISMATCH"))
            .withColumnRenamed("_source_system", "data_source")
            .withColumn("_gold_processed_at", F.current_timestamp())
            .drop("_ingestion_timestamp", "_silver_processed_at", "_dq_warnings",
              "cancellation_code", "crs_dep_hour", "dep_delay_min_pos", "arr_delay_min_pos")
            .withColumn("route", F.concat_ws("-", F.col("origin"), F.col("dest")))
            .withColumn("departure_time_block",
            F.when(F.floor(F.col("crs_dep_time_hhmm") / 100).between(0, 5), "Early Morning")
             .when(F.floor(F.col("crs_dep_time_hhmm") / 100).between(6, 11), "Morning")
             .when(F.floor(F.col("crs_dep_time_hhmm") / 100).between(12, 16), "Afternoon")
             .when(F.floor(F.col("crs_dep_time_hhmm") / 100).between(17, 20), "Evening")
             .when(F.floor(F.col("crs_dep_time_hhmm") / 100).between(21, 23), "Night")
             .otherwise(None)
        )
        .withColumn("haul_length", F.when(F.col("distance_mi") < 500, "Short-haul")
                    .when(F.col("distance_mi") < 1500, "Medium-haul").otherwise("Long-haul"))
    )

# 3-year batches, 2004-2026. Adjust this list if a batch fails partway and you need to resume.
BATCHES = [(2004, 2006), (2007, 2009), (2010, 2012), (2013, 2015),(2016, 2018), 
           (2019, 2021), (2022, 2024), (2025, 2026),]

for start_year, end_year in BATCHES:
    print(f"Processing {start_year}-{end_year}...")

    batch_df = spark.table("silver.flights").filter(
        f"year(flight_date) BETWEEN {start_year} AND {end_year}"
    )
    result_df = build_fct_flights(batch_df)

    (result_df.write.format("delta").mode("overwrite")
     .option("replaceWhere", f"year >= {start_year} AND year <= {end_year}")
     .saveAsTable("gold.fct_flights")
    )

    print(f"  Done: {result_df.count():,} rows written")

##### Step 5: Reconcile gold.fct_flights against silver.flights

In [0]:
silver_count = spark.sql("SELECT COUNT(*) AS cnt FROM silver.flights").collect()[0]["cnt"]
gold_count = spark.sql("SELECT COUNT(*) AS cnt FROM gold.fct_flights").collect()[0]["cnt"]

print(f"silver.flights row count: {silver_count:,}")
print(f"gold.fct_flights row count: {gold_count:,}")
print(f"Match: {silver_count == gold_count}")

# Year-by-year check, to catch any batch-boundary gaps or overlaps
spark.sql("""
    SELECT 'silver' AS layer, year, COUNT(*) AS row_count FROM silver.flights GROUP BY year
    UNION ALL
    SELECT 'gold' AS layer, year, COUNT(*) AS row_count FROM gold.fct_flights GROUP BY year
    ORDER BY year, layer
""").show(50)

### Building Dimension Table

##### Step 6a: Install the holidays package (needed for dim_date)

In [0]:
# %pip install holidays

##### Step 6b: Build dim_date (2004-01-01 through 2026-12-31)

In [0]:
import holidays
import pandas as pd
from pyspark.sql.types import (StructType, StructField, DateType, IntegerType, StringType, BooleanType)

# Generate the full calendar range as a Pandas date range first —
# simplest way to get every single calendar day with no gaps
date_range = pd.date_range(start="2004-01-01", end="2026-12-31", freq="D")

us_holidays = holidays.US(years=range(2004, 2027))

rows = []
for d in date_range:
    py_date = d.date()
    rows.append({"date_key": py_date, "year": d.year,"quarter": (d.month - 1) // 3 + 1,
        "month": d.month,"month_name": d.strftime("%B"),"day_of_month": d.day,
        "day_of_week": d.isoweekday(),          # 1=Monday .. 7=Sunday, matches fct_flights
        "day_name": d.strftime("%A"), "is_weekend": d.isoweekday() in (6, 7),
        "is_us_federal_holiday": py_date in us_holidays, "holiday_name": us_holidays.get(py_date),
    })

pdf = pd.DataFrame(rows)

schema = StructType([
    StructField("date_key", DateType(), False),
    StructField("year", IntegerType(), False),
    StructField("quarter", IntegerType(), False),
    StructField("month", IntegerType(), False),
    StructField("month_name", StringType(), False),
    StructField("day_of_month", IntegerType(), False),
    StructField("day_of_week", IntegerType(), False),
    StructField("day_name", StringType(), False),
    StructField("is_weekend", BooleanType(), False),
    StructField("is_us_federal_holiday", BooleanType(), False),
    StructField("holiday_name", StringType(), True),
])

dim_date = spark.createDataFrame(pdf, schema=schema)

print(f"Row count: {dim_date.count():,}")  


In [0]:
dim_date.filter("is_us_federal_holiday = true").show(15, truncate=False)

 ##### Step 6c: Write dim_date as a managed Delta table

In [0]:
(dim_date.write.format("delta").mode("overwrite").saveAsTable("gold.dim_date"))

spark.sql("SELECT COUNT(*) AS row_count FROM gold.dim_date").show()
spark.sql("DESCRIBE TABLE gold.dim_date").show(20, truncate=False)

##### Step 7a: Distinct carrier codes, with first/last year seen and flight volume

In [0]:
spark.sql("""
    SELECT
        carrier_code,
        MIN(year) AS first_year,
        MAX(year) AS last_year,
        COUNT(*) AS flight_count
    FROM gold.fct_flights
    GROUP BY carrier_code
    ORDER BY carrier_code
""").show(100, truncate=False)

##### Step 7b: Confirm code reuse via DOT_ID (2009+ only, where DOT_ID is available)

In [0]:
spark.sql("""
    SELECT
        Reporting_Airline AS carrier_code,
        COUNT(DISTINCT DOT_ID_Reporting_Airline) AS distinct_dot_ids,
        COLLECT_SET(DOT_ID_Reporting_Airline) AS dot_ids
    FROM bronze.flights_recent
    GROUP BY Reporting_Airline
    HAVING COUNT(DISTINCT DOT_ID_Reporting_Airline) > 1
    ORDER BY carrier_code
""").show(truncate=False)

##### Step 7c: Find the OH DOT_ID cutover year

In [0]:
spark.sql("""
    SELECT
        Year,
        DOT_ID_Reporting_Airline AS dot_id,
        COUNT(*) AS flight_count,
        MIN(FlightDate) AS first_date,
        MAX(FlightDate) AS last_date
    FROM bronze.flights_recent
    WHERE Reporting_Airline = 'OH'
    GROUP BY Year, DOT_ID_Reporting_Airline
    ORDER BY Year, dot_id
""").show(30, truncate=False)

##### Step 7d: Build dim_carrier with time-bounded handling for the OH reuse case

In [0]:
from pyspark.sql import Row
from datetime import date

# Open-ended range for all codes except OH
OPEN_START = date(1900, 1, 1)
OPEN_END = date(9999, 12, 31)

carrier_rows = [
    ("9E", "Endeavor Air", OPEN_START, OPEN_END),
    ("AA", "American Airlines", OPEN_START, OPEN_END),
    ("AQ", "Aloha Airlines", OPEN_START, OPEN_END),
    ("AS", "Alaska Airlines", OPEN_START, OPEN_END),
    ("B6", "JetBlue Airways", OPEN_START, OPEN_END),
    ("CO", "Continental Airlines", OPEN_START, OPEN_END),
    ("DH", "Independence Air", OPEN_START, OPEN_END),
    ("DL", "Delta Air Lines", OPEN_START, OPEN_END),
    ("EV", "ExpressJet Airlines", OPEN_START, OPEN_END),
    ("F9", "Frontier Airlines", OPEN_START, OPEN_END),
    ("FL", "AirTran Airways", OPEN_START, OPEN_END),
    ("G4", "Allegiant Air", OPEN_START, OPEN_END),
    ("HA", "Hawaiian Airlines", OPEN_START, OPEN_END),
    ("HP", "America West Airlines", OPEN_START, OPEN_END),
    ("MQ", "Envoy Air", OPEN_START, OPEN_END),
    ("NK", "Spirit Airlines", OPEN_START, OPEN_END),
    ("NW", "Northwest Airlines", OPEN_START, OPEN_END),
    ("OH", "Comair", OPEN_START, date(2010, 12, 31)),
    ("OH", "PSA Airlines", date(2018, 1, 1), OPEN_END),
    ("OO", "SkyWest Airlines", OPEN_START, OPEN_END),
    ("QX", "Horizon Air", OPEN_START, OPEN_END),
    ("TZ", "ATA Airlines", OPEN_START, OPEN_END),
    ("UA", "United Airlines", OPEN_START, OPEN_END),
    ("US", "US Airways", OPEN_START, OPEN_END),
    ("VX", "Virgin America", OPEN_START, OPEN_END),
    ("WN", "Southwest Airlines", OPEN_START, OPEN_END),
    ("XE", "ExpressJet Airlines (Continental Express)", OPEN_START, OPEN_END),
    ("YV", "Mesa Airlines", OPEN_START, OPEN_END),
    ("YX", "Republic Airways", OPEN_START, OPEN_END),
]

dim_carrier = spark.createDataFrame(
    carrier_rows,
    schema=["carrier_code", "carrier_name", "effective_start", "effective_end"]
)

print(f"Row count: {dim_carrier.count()}")  # expect 29 (28 codes + 1 extra OH row)
dim_carrier.filter("carrier_code = 'OH'").show(truncate=False)

##### Step 7e: Write dim_carrier as a managed Delta table

In [0]:
(dim_carrier.write.format("delta").mode("overwrite").saveAsTable("gold.dim_carrier"))

spark.sql("SELECT COUNT(*) AS row_count FROM gold.dim_carrier").show()
spark.sql("DESCRIBE TABLE gold.dim_carrier").show(truncate=False)

##### Step 8a: Check for airport codes present only in pre-2009 data

In [0]:


pre2009_airports = (spark.table("gold.fct_flights").filter("year <= 2008")
                    .select(F.col("origin").alias("code"))
                    .union(spark.table("gold.fct_flights").filter("year <= 2008").select(F.col("dest").alias("code")))
                    .distinct()
)

post2009_airports = (spark.table("gold.fct_flights").filter("year >= 2009")
    .select(F.col("origin").alias("code"))
    .union(spark.table("gold.fct_flights").filter("year >= 2009").select(F.col("dest").alias("code")))
    .distinct()
)

only_pre2009 = pre2009_airports.subtract(post2009_airports)
print(f"Airport codes only in pre-2009 data: {only_pre2009.count()}")
only_pre2009.show(50, truncate=False)

##### Step 8b: Build dim_airport from bronze.flights_recent (rich metadata, 2009+)

City/state casing normalized with `initcap` before dedup — the raw source has
inconsistent casing (e.g. `"CONCORD, NC"` vs `"Concord, NC"`) that otherwise
shows up as false conflicting-metadata rows for the same airport.


In [0]:
dim_airport_main = (spark.table("bronze.flights_recent")
    .select(F.col("Origin").alias("airport_code"),
        F.col("OriginAirportID").alias("airport_id"),
        F.initcap(F.col("OriginCityName")).alias("city_name"),
        F.col("OriginState").alias("state_code"),
        F.col("OriginStateName").alias("state_name"),
    )
    .union(spark.table("bronze.flights_recent")
        .select(F.col("Dest").alias("airport_code"),
            F.col("DestAirportID").alias("airport_id"),
            F.initcap(F.col("DestCityName")).alias("city_name"),
            F.col("DestState").alias("state_code"),
            F.col("DestStateName").alias("state_name"),
        )
    )
    .distinct()
    .withColumn("data_coverage", F.lit("full_history"))
)

print(f"Row count: {dim_airport_main.count()}")

# Sanity check: are there any airport codes with more than one row (conflicting metadata)?
dupes = (dim_airport_main.groupBy("airport_code").count().filter("count > 1"))
print(f"Airport codes with conflicting metadata: {dupes.count()}")
dupes.show(20)

##### Step 8d: Manual patch for the 17 pre-2009-only airport codes
No OriginAirportID available for these (only present in BTS 2009+ source),so airport_id is NULL here — flagged via data_coverage instead.

In [0]:

patch_schema = StructType([
    StructField("airport_code", StringType(), True),
    StructField("airport_id", IntegerType(), True),
    StructField("city_name", StringType(), True),
    StructField("state_code", StringType(), True),
    StructField("state_name", StringType(), True),
])

patch_rows = [
    ("VIS", None, "Visalia",              "CA", "California"),
    ("ILE", None, "Killeen",              "TX", "Texas"),
    ("HKY", None, "Hickory",              "NC", "North Carolina"),
    ("ISO", None, "Kinston",              "NC", "North Carolina"),
    ("DUT", None, "Unalaska",             "AK", "Alaska"),
    ("APF", None, "Naples",               "FL", "Florida"),
    ("LNY", None, "Lanai City",           "HI", "Hawaii"),
    ("MKK", None, "Kaunakakai",           "HI", "Hawaii"),
    ("SOP", None, "Pinehurst/Southern Pines", "NC", "North Carolina"),
    ("MTH", None, "Marathon",             "FL", "Florida"),
    ("SLE", None, "Salem",                "OR", "Oregon"),
    ("GLH", None, "Greenville",           "MS", "Mississippi"),
    ("PMD", None, "Palmdale",             "CA", "California"),
    ("MKC", None, "Kansas City",          "MO", "Missouri"),
    ("CBM", None, "Columbus",             "MS", "Mississippi"),
    ("RCA", None, "Rapid City",           "SD", "South Dakota"),
    ("SKA", None, "Spokane",              "WA", "Washington"),
]

dim_airport_patch = spark.createDataFrame(patch_rows,schema=patch_schema).withColumn("data_coverage", F.lit("pre_2009_only"))

print(f"Patch row count: {dim_airport_patch.count()}")  # expect 17
dim_airport_patch.show(truncate=False)

##### Step 8d: Combine main + patch, write gold.dim_airport

In [0]:
dim_airport = dim_airport_main.unionByName(dim_airport_patch)

print(f"Final row count: {dim_airport.count()}")  # expect 426 (409 + 17)

# Sanity check: no duplicate codes across the union
dupes = dim_airport.groupBy("airport_code").count().filter("count > 1")
print(f"Duplicate codes after union: {dupes.count()}")

In [0]:
(
    dim_airport
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("gold.dim_airport")
)

spark.sql("SELECT COUNT(*) AS row_count FROM gold.dim_airport").show()
spark.sql("DESCRIBE TABLE gold.dim_airport").show(truncate=False)

### Build Aggregate Tables

##### Step 9 (revised): Coalesce delay-cause sums to 0 instead of NULL
The five delay-cause sums are coalesced to `0` rather than left `NULL` — `SUM()`
over an all-null group returns `NULL`, which in an aggregate table reads as
"unknown" when it should read as "zero delay-cause minutes occurred here."

In [0]:
agg_daily = (
    spark.table("gold.fct_flights")
    .groupBy("flight_date", "carrier_code", "origin", "dest", "route")
    .agg(
        F.count("*").alias("flight_count"),
        F.sum(F.when(~F.col("is_cancelled") & ~F.col("is_diverted"), 1).otherwise(0)).alias("completed_count"),
        F.sum(F.col("is_cancelled").cast("int")).alias("cancelled_count"),
        F.sum(F.col("is_diverted").cast("int")).alias("diverted_count"),
        F.avg(F.when(~F.col("is_cancelled"), F.col("dep_delay_min"))).alias("avg_dep_delay_min"),
        F.avg(F.when(~F.col("is_cancelled"), F.col("arr_delay_min"))).alias("avg_arr_delay_min"),
        F.sum(F.col("dep_del15").cast("int")).alias("dep_del15_count"),
        F.sum(F.col("arr_del15").cast("int")).alias("arr_del15_count"),
        F.avg("taxi_out_min").alias("avg_taxi_out_min"),
        F.avg("taxi_in_min").alias("avg_taxi_in_min"),
        F.sum("distance_mi").alias("total_distance_mi"),
        F.coalesce(F.sum("carrier_delay_min"), F.lit(0)).alias("carrier_delay_min"),
        F.coalesce(F.sum("weather_delay_min"), F.lit(0)).alias("weather_delay_min"),
        F.coalesce(F.sum("nas_delay_min"), F.lit(0)).alias("nas_delay_min"),
        F.coalesce(F.sum("security_delay_min"), F.lit(0)).alias("security_delay_min"),
        F.coalesce(F.sum("late_aircraft_delay_min"), F.lit(0)).alias("late_aircraft_delay_min"),
    )
    .withColumn("_gold_processed_at", F.current_timestamp())
)

print(f"Row count: {agg_daily.count():,}")  # should be unchanged: 52,154,788
agg_daily.filter("carrier_code = 'WN' AND origin = 'DEN' AND dest = 'TUL'").show(truncate=False)

##### Step 9b: Write the aggregate table with clustering for dashboard query patterns

In [0]:
(
    agg_daily
    .write
    .format("delta")
    .mode("overwrite")
    .clusterBy("flight_date", "carrier_code")
    .saveAsTable("gold.agg_daily_carrier_route_performance")
)

result = spark.sql("SELECT COUNT(*) AS row_count FROM gold.agg_daily_carrier_route_performance")
result.show()

spark.sql("DESCRIBE TABLE gold.agg_daily_carrier_route_performance").show(30, truncate=False)

##### Step 10: Reconcile agg_daily_carrier_route_performance against fct_flights

In [0]:
fct_total = spark.sql("SELECT COUNT(*) AS cnt FROM gold.fct_flights").collect()[0]["cnt"]
agg_total = spark.sql("SELECT SUM(flight_count) AS cnt FROM gold.agg_daily_carrier_route_performance").collect()[0]["cnt"]

print(f"fct_flights row count:              {fct_total:,}")
print(f"agg SUM(flight_count):              {agg_total:,}")
print(f"Match: {fct_total == agg_total}")

# Spot-check cancelled/diverted/completed totals too, not just flight_count
spark.sql("""
    SELECT
        (SELECT COUNT(*) FROM gold.fct_flights WHERE is_cancelled) AS fct_cancelled,
        (SELECT SUM(cancelled_count) FROM gold.agg_daily_carrier_route_performance) AS agg_cancelled,
        (SELECT COUNT(*) FROM gold.fct_flights WHERE is_diverted) AS fct_diverted,
        (SELECT SUM(diverted_count) FROM gold.agg_daily_carrier_route_performance) AS agg_diverted,
        (SELECT SUM(carrier_delay_min) FROM gold.fct_flights) AS fct_carrier_delay,
        (SELECT SUM(carrier_delay_min) FROM gold.agg_daily_carrier_route_performance) AS agg_carrier_delay
""").show(truncate=False)

##### Step 11: Build gold.agg_monthly_carrier_performance

In [0]:
agg_monthly = (
    spark.table("gold.fct_flights")
    .groupBy("year", "month", "carrier_code")
    .agg(
        F.count("*").alias("flight_count"),
        F.sum(F.when(~F.col("is_cancelled") & ~F.col("is_diverted"), 1).otherwise(0)).alias("completed_count"),
        F.sum(F.col("is_cancelled").cast("int")).alias("cancelled_count"),
        F.sum(F.col("is_diverted").cast("int")).alias("diverted_count"),
        F.avg(F.when(~F.col("is_cancelled"), F.col("dep_delay_min"))).alias("avg_dep_delay_min"),
        F.avg(F.when(~F.col("is_cancelled"), F.col("arr_delay_min"))).alias("avg_arr_delay_min"),
        F.sum(F.col("dep_del15").cast("int")).alias("dep_del15_count"),
        F.sum(F.col("arr_del15").cast("int")).alias("arr_del15_count"),
        F.avg("taxi_out_min").alias("avg_taxi_out_min"),
        F.avg("taxi_in_min").alias("avg_taxi_in_min"),
        F.sum("distance_mi").alias("total_distance_mi"),
        F.coalesce(F.sum("carrier_delay_min"), F.lit(0)).alias("carrier_delay_min"),
        F.coalesce(F.sum("weather_delay_min"), F.lit(0)).alias("weather_delay_min"),
        F.coalesce(F.sum("nas_delay_min"), F.lit(0)).alias("nas_delay_min"),
        F.coalesce(F.sum("security_delay_min"), F.lit(0)).alias("security_delay_min"),
        F.coalesce(F.sum("late_aircraft_delay_min"), F.lit(0)).alias("late_aircraft_delay_min"),
    )
    .withColumn("_gold_processed_at", F.current_timestamp())
)

print(f"Row count: {agg_monthly.count():,}")  # expect roughly 23 years * 12 months * ~28 carriers, minus gaps where a carrier didn't operate that month
agg_monthly.orderBy("year", "month", "carrier_code").show(10, truncate=False)

##### Step 11b: Write the monthly summary table


In [0]:
(
    agg_monthly
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("gold.agg_monthly_carrier_performance")
)

spark.sql("SELECT COUNT(*) AS row_count FROM gold.agg_monthly_carrier_performance").show()
spark.sql("DESCRIBE TABLE gold.agg_monthly_carrier_performance").show(30, truncate=False)

##### Step 11c: Reconcile monthly aggregate against fct_flights

In [0]:
spark.sql("""
    SELECT
        (SELECT COUNT(*) FROM gold.fct_flights) AS fct_total,
        (SELECT SUM(flight_count) FROM gold.agg_monthly_carrier_performance) AS monthly_total,
        (SELECT SUM(carrier_delay_min) FROM gold.fct_flights) AS fct_carrier_delay,
        (SELECT SUM(carrier_delay_min) FROM gold.agg_monthly_carrier_performance) AS monthly_carrier_delay
""").show(truncate=False)